In [59]:
from musicProcessing import MusicProcessingProvider as mp
from alg_bins import AlgBins

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

In [60]:
# mp params
path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/babyslakh_16k/Track00001/mix.wav"
duration = None
nfft = 2048
win_length = 2048
hop_length = 1024
window_type = "hann"

music_processor = mp(path, duration, nfft, hop_length, win_length, window_type)

In [ ]:
# stable alg params
num_generations = 100
population_size = 200
sharing_radius = 0.5
mutation_probability = 0.15
crossover_probability = 0.7
population_initialization_method = "random" # "random" or "elite"
selection_method = "tournament" # "tournament" or "roulette"
Individual_bin_count = 20

In [ ]:
similarity_metrics = ["jaccard", "cosine", "fenotype"]
crossover_methods = ["single_point", "two_point"]
random_seeds = np.arange(20)

results_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/experiments/results/exp1/"

In [63]:
global_df = pd.DataFrame(columns=["similarity_metric", "crossover_method", "best_end_fitness", "best_end_fitness_std", "best_end_shared_fitness", "best_end_shared_fitness_std", "avg_end_similarity", "avg_end_similarity_std"])

In [64]:
for similarity_metric in similarity_metrics:
    for crossover_method in crossover_methods:
        # rng independent values
        best_fitness = np.zeros((num_generations, len(random_seeds)))
        best_shared_fitness = np.zeros((num_generations, len(random_seeds)))
        avg_similarity = np.zeros((num_generations, len(random_seeds)))
        for random_seed in random_seeds:
            print(f"Running experiment with similarity metric: {similarity_metric}, crossover method: {crossover_method}, random seed: {random_seed}")
            alg = AlgBins(mp=music_processor, random_seed=random_seed, num_generations=num_generations, population_size=population_size, individual_bin_count=Individual_bin_count, mutation_probability=mutation_probability, crossover_probability=crossover_probability, crossover_method=crossover_method, selection_method=selection_method, sim_metric=similarity_metric, sharing_radius=sharing_radius, population_initialization_method=population_initialization_method)
            alg.run()
            
            best_fitness[:, random_seed] = alg.best_fitness    
            best_shared_fitness[:, random_seed] = alg.best_shared_fitness
            avg_similarity[:, random_seed] = alg.avg_similarity
            
        # plot stored metrics
        plt.figure(figsize=(14, 8))
        plt.subplot(2, 2, 1)
        plt.plot(avg_similarity.mean(axis=1), label="Similarity")
        plt.xlabel("Generation")
        plt.ylabel("Similarity")
        plt.title("Population Similarity Over Generations")
        plt.legend()
        plt.subplot(2, 2, 2)
        plt.plot(best_shared_fitness.mean(axis=1), label="Best")
        plt.xlabel("Generation")
        plt.ylabel("Shared Fitness")
        plt.title("Best Shared Fitness Over Generations")
        plt.legend()
        plt.subplot(2, 2, 3)
        plt.plot(best_fitness.mean(axis=1), label="Best")
        plt.xlabel("Generation")
        plt.ylabel("Fitness")
        plt.title("Best Fitness Over Generations")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(results_path, f"similarity_{similarity_metric}_crossover_{crossover_method}.png"))
        plt.close()
        
        # store results in dataframe
        row = pd.DataFrame([{
            "similarity_metric": similarity_metric,
            "crossover_method": crossover_method,
            "best_end_fitness": best_fitness[-1, :].mean(),
            "best_end_fitness_std": best_fitness[-1, :].std(),
            "best_end_shared_fitness": best_shared_fitness[-1, :].mean(),
            "best_end_shared_fitness_std": best_shared_fitness[-1, :].std(),
            "avg_end_similarity": avg_similarity[-1, :].mean(),
            "avg_end_similarity_std": avg_similarity[-1, :].std()
        }])
        global_df = pd.concat([global_df, row], ignore_index=True)

global_df.to_csv(os.path.join(results_path, "results.csv"), index=False)

Running experiment with similarity metric: jaccard, crossover method: single_point, random seed: 0


Generations: 100%|██████████| 10/10 [00:00<00:00, 12.62it/s]


Running experiment with similarity metric: jaccard, crossover method: two_point, random seed: 0


Generations: 100%|██████████| 10/10 [00:00<00:00, 12.19it/s]


Running experiment with similarity metric: cosine, crossover method: single_point, random seed: 0


...

Generations: 100%|██████████| 10/10 [00:00<00:00, 13.47it/s]


Running experiment with similarity metric: fenotype, crossover method: single_point, random seed: 0


Generations: 100%|██████████| 10/10 [00:02<00:00,  4.50it/s]


Running experiment with similarity metric: fenotype, crossover method: two_point, random seed: 0


Generations: 100%|██████████| 10/10 [00:02<00:00,  4.65it/s]
